# Tugas Besar 2 IF3170: Prediksi Pembatalan Pemesanan Penginapan

# Initialization

## Import Libraries

In [1]:
import time

import numpy as np
import pandas as pd

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else None
except ImportError:
    torch, DEVICE = None, None

np.random.seed(42)
print('GPU untuk KNN:', DEVICE)

GPU untuk KNN: cuda


## Settings

In [2]:
SEEDS = [1, 2, 3]          # seed fold buat OOF, skornya rata-rata semua seed
CHECK_SEEDS = [4, 5, 42]    # seed lain buat cek akhir
MIN_GAIN = 2e-4             # OOF harus naik minimal segini biar opsi dipakai
PASSES = 2
SUB_PATH = 'submission_knn_ladder.csv'

## Load Dataset

In [3]:
train = pd.read_csv('train.csv', dtype={'id': str})
test = pd.read_csv('test.csv', dtype={'id': str})
pd.Series({'train': len(train), 'test': len(test)})

,0
train,27373
test,6844


# Kode From Scratch

## Feature dan Cleaning

In [4]:
TARGET = 'is_canceled'
ID_COL = 'id'
CAT_COLS = ['meal_package', 'room_category', 'booking_channel']
RAW = tuple(c for c in train.columns if c not in (ID_COL, TARGET))
ENGINEERED = ('total_nights', 'total_guests', 'is_free', 'rate_per_guest', 'arrival_weekday',
              'arrival_dayofyear', 'arrival_index', 'booking_index', 'doy_sin', 'doy_cos')
NEW = ('weekend_ratio', 'total_cost', 'has_children', 'is_weekend_arrival', 'month_sin', 'month_cos')
POOL = RAW + ENGINEERED + NEW

FEATURE_SETS = {
    'raw': RAW,
    'raw+fe': RAW + ENGINEERED,
    'v2': ('days_before_arrival', 'arrival_index', 'special_requests', 'nightly_rate_idr',
           'booking_channel', 'arrival_dayofyear', 'total_nights', 'needs_parking'),
    'combo1347': ('days_before_arrival', 'arrival_index', 'special_requests', 'nightly_rate_idr',
                  'booking_channel', 'doy_sin', 'doy_cos', 'total_nights', 'needs_parking', 'booking_index'),
}

CLIP_COLS = ['num_adults', 'num_children', 'weekend_nights', 'weekday_nights', 'days_before_arrival',
             'nightly_rate_idr', 'prior_cancellations', 'prior_completed_bookings', 'special_requests']
LOG_COLS = {'days_before_arrival', 'nightly_rate_idr', 'prior_cancellations', 'prior_completed_bookings',
            'weekend_nights', 'weekday_nights', 'total_nights', 'rate_per_guest', 'total_cost'}
ORD_MAPS = {'meal_package': {'Room Only': 0, 'Breakfast': 1, 'Half Board': 2, 'Full Board': 3}}


# Data train setelah cleaning baris: label bertentangan lalu dedupe.
def get_train(cfg):
    df = train.copy()
    feat_cols = [c for c in df.columns if c not in (ID_COL, TARGET)]
    if cfg['conflict'] != 'keep':
        key = df[feat_cols].astype(object).fillna('nan').astype(str).agg('|'.join, axis=1)
        mean = df.groupby(key)[TARGET].transform('mean')
        if cfg['conflict'] == 'drop':
            df = df[(mean == 0) | (mean == 1)]
        else:
            df = df[mean != 0.5].copy()
            df[TARGET] = (mean[mean != 0.5] > 0.5).astype(int)
    if cfg['dedupe']:
        df = df.drop_duplicates(subset=feat_cols + [TARGET])
    return df.reset_index(drop=True)


# Tandai nilai janggal sebagai missing.
def mark_invalid(df, cfg):
    df = df.copy()
    if cfg['invalid']:
        df.loc[df['num_adults'] == 0, 'num_adults'] = np.nan
    if cfg['invalid_extra']:
        zero = (df['weekend_nights'] + df['weekday_nights']) == 0
        df.loc[zero, ['weekend_nights', 'weekday_nights']] = np.nan
        df.loc[df['num_children'] >= 9, 'num_children'] = np.nan
        df.loc[(df['nightly_rate_idr'] == 0) & (df['booking_channel'] != 'Complimentary'), 'nightly_rate_idr'] = np.nan
    return df


# Batas clipping dari data train. Kolom yang batasnya sama tidak dipotong.
def clip_bounds(df, how):
    x = df[CLIP_COLS]
    if how == 'iqr':
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        lo, hi = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)
    else:
        a = {'p99': 0.01, 'p995': 0.005}[how]
        lo, hi = x.quantile(a), x.quantile(1 - a)
    same = lo >= hi
    return lo.where(~same, -np.inf), hi.where(~same, np.inf)


# Semua fitur turunan. Kolom yang dipakai ditentukan oleh cfg['features'].
def engineer(df):
    df = df.copy()
    df['total_nights'] = df['weekend_nights'] + df['weekday_nights']
    df['total_guests'] = df['num_adults'] + df['num_children']
    df['is_free'] = (df['nightly_rate_idr'] == 0).astype(float)
    df.loc[df['nightly_rate_idr'].isna(), 'is_free'] = np.nan
    df['rate_per_guest'] = df['nightly_rate_idr'] / df['total_guests'].replace(0, np.nan)
    arrival = pd.to_datetime(dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']),
                             errors='coerce')
    df['arrival_weekday'] = arrival.dt.day_of_week.astype(float)
    df['arrival_dayofyear'] = arrival.dt.day_of_year.astype(float)
    df['arrival_index'] = (arrival - pd.Timestamp('2023-01-01')).dt.days.astype(float)
    df['booking_index'] = df['arrival_index'] - df['days_before_arrival']
    df['doy_sin'] = np.sin(2 * np.pi * df['arrival_dayofyear'] / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df['arrival_dayofyear'] / 365.25)
    df['weekend_ratio'] = df['weekend_nights'] / df['total_nights'].replace(0, np.nan)
    df['total_cost'] = df['nightly_rate_idr'] * df['total_nights']
    df['has_children'] = (df['num_children'] > 0).astype(float).where(df['num_children'].notna())
    df['is_weekend_arrival'] = (df['arrival_weekday'] >= 5).astype(float).where(df['arrival_weekday'].notna())
    df['month_sin'] = np.sin(2 * np.pi * df['arrival_month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['arrival_month'] / 12)
    return df


# Preprocessing bersama KNN dan DTL. Semua statistik dipelajari dari fit_df saja.
# Hasilnya frame fit dan list frame lain yang sudah diproses.
def common_frames(cfg, fit_df, *frames):
    raw = [mark_invalid(d, cfg) for d in (fit_df,) + frames]
    if cfg['clip']:
        lo, hi = clip_bounds(raw[0], cfg['clip'])
        for d in raw:
            d[CLIP_COLS] = d[CLIP_COLS].clip(lower=lo, upper=hi, axis=1)
    eng = [engineer(d) for d in raw]
    F = eng[0]
    if cfg['rare']:
        for c in CAT_COLS:
            counts = F[c].value_counts()
            keep = set(counts[counts >= cfg['rare']].index)
            for d in eng:
                d.loc[d[c].notna() & ~d[c].isin(keep), c] = 'Other'
    num = [c for c in F.columns if c not in CAT_COLS and c not in (ID_COL, TARGET)]
    if cfg['miss_ind']:
        for c in num:
            if F[c].isna().any():
                for d in eng:
                    d['miss_' + c] = d[c].isna().astype(float)
    if cfg['impute']:
        stat = {'median': F[num].median(), 'mean': F[num].mean(), 'mode': F[num].mode().iloc[0]}[cfg['impute']]
        fill = {**stat.to_dict(), **{c: F[c].mode()[0] for c in CAT_COLS}}
        eng = [d.fillna(fill) for d in eng]
    return eng[0], eng[1:]


# Kode angka untuk kategori: ordinal atau frequency, dipelajari dari frame fit F.
def cat_codes(F, d, c, how):
    if how == 'freq':
        return d[c].map(F[c].value_counts(normalize=True)).astype(float).fillna(0.0)
    mapping = ORD_MAPS.get(c) or {v: i for i, v in enumerate(sorted(F[c].dropna().unique()))}
    return d[c].map(mapping).astype(float)

## Metrik, Fold, dan Ladder

In [5]:
# Macro F1 biner: rata-rata F1 kelas 0 dan kelas 1.
def macro_f1(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1)); tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1)); fn = np.sum((y_true == 1) & (y_pred == 0))
    f1_pos = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0
    f1_neg = 2 * tn / (2 * tn + fn + fp) if tn + fn + fp else 0.0
    return float((f1_pos + f1_neg) / 2)


# Threshold dengan Macro F1 tertinggi pada prediksi OOF.
def best_threshold(y_true, proba):
    grid = np.round(np.arange(0.20, 0.81, 0.01), 2)
    return float(grid[int(np.argmax([macro_f1(y_true, proba >= t) for t in grid]))])


# Stratified k-fold manual, hasilnya list (train_idx, val_idx).
def stratified_kfold(y, n_splits=5, seed=42):
    rng = np.random.RandomState(seed)
    fold_of = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        fold_of[idx] = np.arange(len(idx)) % n_splits
    return [(np.where(fold_of != k)[0], np.where(fold_of == k)[0]) for k in range(n_splits)]


# OOF satu cfg di semua seed, threshold dari gabungan OOF
def evaluate(cfg, fit_predict=None, seeds=None):
    fit_predict = fit_predict or knn_fit_predict
    try:
        tr = get_train(cfg)
        y = tr[TARGET].to_numpy()
        oofs = []
        for seed in (seeds or SEEDS):
            oof = np.zeros(len(y))
            for a, b in stratified_kfold(y, 5, seed):
                oof[b] = fit_predict(cfg, tr.iloc[a], tr.iloc[b])
            oofs.append(oof)
        th = best_threshold(np.tile(y, len(oofs)), np.concatenate(oofs)) if cfg['threshold'] == 'oof' else 0.5
        return {'oof': float(np.mean([macro_f1(y, o >= th) for o in oofs])), 'th': th, 'err': ''}
    except Exception as e:
        return {'oof': np.nan, 'th': np.nan, 'err': f'{type(e).__name__}: {e}'}


CACHE = {}


def run_many(cfgs):
    out = []
    for c in cfgs:
        key = repr(sorted(c.items()))
        if key not in CACHE:
            CACHE[key] = evaluate(c)
        out.append(CACHE[key])
    return out


# opsi grid buat satu parameter.
def grid(key, values):
    return lambda cfg: [(f'{key}={v}', {key: v}) for v in values if v != cfg[key]]


def set_features():
    return lambda cfg: [(f'features={n}', {'features': f}) for n, f in FEATURE_SETS.items() if f != cfg['features']]


def add_feature():
    return lambda cfg: [(f'+{f}', {'features': cfg['features'] + (f,)}) for f in POOL if f not in cfg['features']]


def drop_feature():
    return lambda cfg: [(f'-{f}', {'features': tuple(x for x in cfg['features'] if x != f)})
                        for f in cfg['features'] if len(cfg['features']) > 1]


def weight_feature(mults=(0.5, 2.0)):
    def gen(cfg):
        out = []
        for f in cfg['features']:
            for m in mults:
                w = dict(cfg['fw'])
                w[f] = round(w.get(f, 1.0) * m, 4)
                out.append((f'{f} x{w[f]}', {'fw': tuple(sorted(w.items()))}))
        return out
    return gen


def enn_mode():
    return lambda cfg: [] if not cfg['enn_k'] else [(f'enn_mode={m}', {'enn_mode': m})
                                                     for m in ('drop', 'relabel') if m != cfg['enn_mode']]


# ladder: tiap langkah semua opsi dicoba, opsi dengan OOF tertinggi dipakai kalau naik > MIN_GAIN
# langkah dengan repeat > 1 diulang selama masih ada opsi yang dipakai
def ladder(base_cfg, steps):
    cfg = dict(base_cfg)
    best = run_many([cfg])[0]
    rows = [dict(putaran=0, langkah='Baseline', opsi='-', oof=best['oof'], th=best['th'], dipakai=True, oof_saat_ini=best['oof'])]
    print(f"Baseline  OOF {best['oof']:.4f}")
    for p in range(1, PASSES + 1):
        for name, gen, repeat in steps:
            for _ in range(repeat):
                options = gen(cfg)
                if not options:
                    break
                t = time.time()
                trials = [{**cfg, **upd} for _, upd in options]
                results = run_many(trials)
                oofs = np.array([r['oof'] for r in results], dtype=float)
                i = int(np.nanargmax(oofs)) if not np.isnan(oofs).all() else 0
                keep = oofs[i] > best['oof'] + MIN_GAIN
                for j, ((label, _), r) in enumerate(zip(options, results)):
                    rows.append(dict(putaran=p, langkah=name, opsi=label, oof=r['oof'], th=r['th'],
                                     dipakai=keep and j == i, oof_saat_ini=best['oof']))
                print(f"P{p} {name:22s} {options[i][0]:30s} OOF {oofs[i]:.4f} {'DIPAKAI' if keep else 'dibuang'}"
                      f"  sekarang {max(best['oof'], oofs[i]) if keep else best['oof']:.4f}  ({len(options)} opsi, {time.time() - t:.0f}s)", flush=True)
                pd.DataFrame(rows).to_csv('knn_ladder_log.csv', index=False)
                if not keep:
                    break
                cfg, best = trials[i], results[i]
    return cfg, best, pd.DataFrame(rows)

## KNN

In [6]:
# Matriks KNN untuk fit_df dan frame lain.
def knn_prepare(cfg, fit_df, *frames):
    F, outs = common_frames(cfg, fit_df, *frames)
    feats = list(cfg['features'])
    num = [f for f in feats if f not in CAT_COLS]
    cat = [f for f in feats if f in CAT_COLS]
    ind = [f'miss_{f}' for f in num if f'miss_{f}' in F.columns]
    coded = cat if cfg['cat_enc'] != 'onehot' else []
    w = dict(cfg['fw'])

    def numeric(d):
        X = d[num].astype(float).copy()
        for c in coded:
            X[c] = cat_codes(F, d, c, cfg['cat_enc'])
        if cfg['log']:
            for c in X.columns:
                if c in LOG_COLS:
                    X[c] = np.log1p(X[c].clip(lower=0))
        return X

    XF = numeric(F)
    med = XF.median()
    XF = XF.fillna(med)
    s = cfg['scaler']
    if s == 'zscore':
        center, scale = XF.mean(), XF.std()
    elif s == 'minmax':
        center, scale = XF.min(), XF.max() - XF.min()
    elif s == 'robust':
        center, scale = XF.median(), XF.quantile(0.75) - XF.quantile(0.25)
        scale = scale.where(scale > 0, XF.std())
    else:
        sorted_vals = {c: np.sort(XF[c].to_numpy()) for c in XF.columns}
    result = []
    for d in outs:
        X = numeric(d).fillna(med)
        if s == 'rank':
            Z = pd.DataFrame({c: ((np.searchsorted(sorted_vals[c], X[c], 'left') + np.searchsorted(sorted_vals[c], X[c], 'right'))
                                  / (2 * len(sorted_vals[c])) - 0.5) * np.sqrt(12) for c in X.columns}, index=X.index)
        else:
            Z = (X - center) / scale.replace(0, 1)
        parts = [Z[c].to_numpy()[:, None] * w.get(c, 1.0) for c in Z.columns]
        parts += [d[c].to_numpy()[:, None] * cfg['onehot'] for c in ind]
        if cfg['cat_enc'] == 'onehot':
            for c in cat:
                v = d[c].fillna(F[c].mode()[0]).to_numpy()
                levels = sorted(F[c].dropna().unique())
                parts.append(np.stack([(v == lv) for lv in levels], axis=1) * cfg['onehot'] * w.get(c, 1.0))
        result.append(np.hstack(parts).astype(np.float32))
    return result


# k tetangga terdekat dengan jarak Minkowski, terurut dari yang terdekat.
def kneighbors(X, Q, k, p, batch=256):
    if DEVICE is not None:
        Xt = torch.tensor(X, device=DEVICE)
        ds, ids = [], []
        for s in range(0, len(Q), batch):
            q = torch.tensor(Q[s:s + batch], device=DEVICE)
            D = (q[:, None, :] - Xt[None, :, :]).abs().pow(p).sum(-1)
            d, i = torch.topk(D, k, dim=1, largest=False, sorted=True)
            ds.append(d.pow(1 / p).cpu().numpy()); ids.append(i.cpu().numpy())
        return np.vstack(ds), np.vstack(ids)
    ds, ids = [], []
    for s in range(0, len(Q), batch):
        q = Q[s:s + batch]
        D = np.zeros((len(q), len(X)), np.float32)
        for j in range(X.shape[1]):
            D += np.abs(q[:, j, None] - X[None, :, j]) ** p
        idx = np.argpartition(D, k - 1, axis=1)[:, :k]
        d = np.take_along_axis(D, idx, 1)
        o = np.argsort(d, axis=1, kind='stable')
        ids.append(np.take_along_axis(idx, o, 1)); ds.append(np.take_along_axis(d, o, 1) ** (1 / p))
    return np.vstack(ds), np.vstack(ids)


# Edited Nearest Neighbors: baris yang labelnya beda dari mayoritas k tetangganya ditandai noise.
# Hasilnya (mask baris yang disimpan, label baru).
def enn(X, y, k, p, mode):
    _, idx = kneighbors(X, X, k + 1, p)
    is_self = idx == np.arange(len(X))[:, None]
    keep = ~is_self
    keep[~is_self.any(1), -1] = False
    nb = idx[keep].reshape(len(X), k)
    share = y[nb].mean(1)
    noisy = ((share > 0.5) & (y == 0)) | ((share < 0.5) & (y == 1))
    if mode == 'drop':
        return ~noisy, y
    return np.ones(len(y), bool), np.where(noisy, 1 - y, y)


# Probabilitas kelas 1 dari vote tetangga dengan bobot sesuai cfg['weight'].
def knn_fit_predict(cfg, fit_df, pred_df):
    X, Q = knn_prepare(cfg, fit_df, fit_df, pred_df)
    y = fit_df[TARGET].to_numpy().astype(int)
    if cfg['enn_k']:
        mask, y = enn(X, y, cfg['enn_k'], cfg['p'], cfg['enn_mode'])
        X, y = X[mask], y[mask]
    d, idx = kneighbors(X, Q, cfg['k'], cfg['p'])
    if cfg['weight'] == 'uniform':
        w = np.ones_like(d)
    elif cfg['weight'] == 'inv':
        w = 1 / (d + 1e-6)
    elif cfg['weight'] == 'inv2':
        w = 1 / (d ** 2 + 1e-6)
    else:
        w = np.exp(-2 * (d / (d[:, -1:] + 1e-6)) ** 2)
    return (w * y[idx]).sum(1) / w.sum(1)

# Modeling dan Validation

## 4.1 Ladder

In [7]:
KNN_BASE = dict(dedupe=False, conflict='keep', invalid=False, invalid_extra=False, clip=None, impute='median',
                miss_ind=False, rare=0, cat_enc='onehot', onehot=1.0, log=False, scaler='zscore',
                features=FEATURE_SETS['raw'], fw=(), k=5, p=2.0, weight='uniform',
                enn_k=0, enn_mode='drop', threshold='0.5')
KNN_STEPS = [
    ('Dedupe train', grid('dedupe', [True]), 1),
    ('Label bertentangan', grid('conflict', ['majority', 'drop']), 1),
    ('Mark invalid', grid('invalid', [True]), 1),
    ('Nilai janggal lain', grid('invalid_extra', [True]), 1),
    ('Jumlah tetangga k', grid('k', [5, 7, 9, 11, 13, 15, 19, 25, 31, 51]), 1),
    ('Metrik jarak p', grid('p', [0.5, 0.75, 1.0, 1.5, 2.0]), 1),
    ('Bobot vote', grid('weight', ['uniform', 'inv', 'inv2', 'gauss']), 1),
    ('Clipping', grid('clip', ['p99', 'p995', 'iqr']), 1),
    ('Imputasi', grid('impute', ['median', 'mean', 'mode']), 1),
    ('Indikator missing', grid('miss_ind', [True]), 1),
    ('Kategori langka', grid('rare', [20, 100]), 1),
    ('Encoding kategori', grid('cat_enc', ['onehot', 'ordinal', 'freq']), 1),
    ('Skala one-hot', grid('onehot', [0.25, 0.5, 0.75, 1.0]), 1),
    ('Log transform', grid('log', [True]), 1),
    ('Scaler', grid('scaler', ['zscore', 'minmax', 'robust', 'rank']), 1),
    ('Set fitur', set_features(), 1),
    ('Tambah fitur', add_feature(), 4),
    ('Buang fitur', drop_feature(), 4),
    ('Bobot fitur', weight_feature(), 3),
    ('ENN k', grid('enn_k', [3, 5, 7, 11, 15]), 1),
    ('ENN mode', enn_mode(), 1),
    ('Threshold OOF', grid('threshold', ['oof']), 1),
]

In [8]:
t0 = time.time()
knn_cfg, knn_best, knn_log = ladder(KNN_BASE, KNN_STEPS)
print(f'selesai {time.time() - t0:.0f}s')
print({k: v for k, v in knn_cfg.items() if KNN_BASE[k] != v})

Baseline  OOF 0.8120
P1 Dedupe train           dedupe=True                    OOF 0.8098 dibuang  sekarang 0.8120  (1 opsi, 11s)


/tmp/ipykernel_443/359973145.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  key = df[feat_cols].astype(object).fillna('nan').astype(str).agg('|'.join, axis=1)
/tmp/ipykernel_443/359973145.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  key = df[feat_cols].astype(object).fillna('nan').astype(str).agg('|'.join, axis=1)


P1 Label bertentangan     conflict=majority              OOF 0.8104 dibuang  sekarang 0.8120  (2 opsi, 22s)
P1 Mark invalid           invalid=True                   OOF 0.8119 dibuang  sekarang 0.8120  (1 opsi, 11s)
P1 Nilai janggal lain     invalid_extra=True             OOF 0.8112 dibuang  sekarang 0.8120  (1 opsi, 11s)
P1 Jumlah tetangga k      k=7                            OOF 0.8099 dibuang  sekarang 0.8120  (9 opsi, 96s)
P1 Metrik jarak p         p=0.75                         OOF 0.8181 DIPAKAI  sekarang 0.8181  (4 opsi, 44s)
P1 Bobot vote             weight=inv                     OOF 0.8307 DIPAKAI  sekarang 0.8307  (3 opsi, 32s)
P1 Clipping               clip=iqr                       OOF 0.8307 dibuang  sekarang 0.8307  (3 opsi, 34s)
P1 Imputasi               impute=mode                    OOF 0.8271 dibuang  sekarang 0.8307  (2 opsi, 22s)
P1 Indikator missing      miss_ind=True                  OOF 0.8246 dibuang  sekarang 0.8307  (1 opsi, 14s)
P1 Kategori langka        ra

/tmp/ipykernel_443/359973145.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  key = df[feat_cols].astype(object).fillna('nan').astype(str).agg('|'.join, axis=1)
/tmp/ipykernel_443/359973145.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  key = df[feat_cols].astype(object).fillna('nan').astype(str).agg('|'.join, axis=1)


P2 Label bertentangan     conflict=majority              OOF 0.8395 dibuang  sekarang 0.8427  (2 opsi, 14s)
P2 Mark invalid           invalid=True                   OOF 0.8427 dibuang  sekarang 0.8427  (1 opsi, 7s)
P2 Nilai janggal lain     invalid_extra=True             OOF 0.8418 dibuang  sekarang 0.8427  (1 opsi, 7s)
P2 Jumlah tetangga k      k=13                           OOF 0.8461 DIPAKAI  sekarang 0.8461  (9 opsi, 60s)
P2 Metrik jarak p         p=1.0                          OOF 0.8456 dibuang  sekarang 0.8461  (4 opsi, 26s)
P2 Bobot vote             weight=inv2                    OOF 0.8460 dibuang  sekarang 0.8461  (3 opsi, 20s)
P2 Clipping               clip=iqr                       OOF 0.8462 dibuang  sekarang 0.8461  (3 opsi, 22s)
P2 Imputasi               impute=mode                    OOF 0.8442 dibuang  sekarang 0.8461  (2 opsi, 13s)
P2 Indikator missing      miss_ind=True                  OOF 0.8433 dibuang  sekarang 0.8461  (1 opsi, 9s)
P2 Kategori langka        rare=

## 4.2 Langkah yang Dipakai

In [9]:
pd.set_option('display.max_rows', 400)
knn_log[knn_log['dipakai']].round(4)

,putaran,langkah,opsi,oof,th,dipakai,oof_saat_ini
0,0,Baseline,-,0.8120,0.50,True,0.8120
16,1,Metrik jarak p,p=0.75,0.8181,0.50,True,0.8120
19,1,Bobot vote,weight=inv,0.8307,0.50,True,0.8181
33,1,Skala one-hot,onehot=0.5,0.8324,0.50,True,0.8307
37,1,Scaler,scaler=robust,0.8345,0.50,True,0.8324
41,1,Set fitur,features=combo1347,0.8374,0.50,True,0.8345
44,1,Tambah fitur,+weekend_nights,0.8392,0.50,True,0.8374
73,1,Tambah fitur,+is_returning_guest,0.8396,0.50,True,0.8392
109,1,Buang fitur,-arrival_index,0.8403,0.50,True,0.8396
143,1,Bobot fitur,total_nights x0.5,0.8418,0.50,True,0.8403


## 4.3 Cek dengan Pembagian Fold Lain

In [10]:
base_cfg = dict(KNN_BASE, dedupe=True, k=19, p=1.0, weight='inv', onehot=0.5, features=FEATURE_SETS['v2'], threshold='oof')
rows = []
for seed in CHECK_SEEDS:
    rows.append({'seed': seed, 'knn_selection_distance': evaluate(base_cfg, seeds=[seed])['oof'],
                 'knn_ladder': evaluate(knn_cfg, seeds=[seed])['oof']})
check = pd.DataFrame(rows).set_index('seed')
check['selisih'] = check['knn_ladder'] - check['knn_selection_distance']
print('knn_ladder unggul di', int((check['selisih'] > 0).sum()), 'dari', len(check), 'seed')
check.round(4)

knn_ladder unggul di 3 dari 3 seed


,knn_selection_distance,knn_ladder,selisih
seed,,,
4,0.8418,0.8478,0.0060
5,0.8418,0.8475,0.0057
42,0.8411,0.8483,0.0072


# Submission

In [11]:
tr = get_train(knn_cfg)
THRESHOLD = knn_best['th']
proba = knn_fit_predict(knn_cfg, tr, test)
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: (proba >= THRESHOLD).astype(int)})
submission.to_csv(SUB_PATH, index=False)
print('CONFIG =', knn_cfg)
print('THRESHOLD =', THRESHOLD)
submission[TARGET].value_counts(normalize=True).round(4)

CONFIG = {'dedupe': False, 'conflict': 'keep', 'invalid': False, 'invalid_extra': False, 'clip': None, 'impute': 'median', 'miss_ind': False, 'rare': 0, 'cat_enc': 'ordinal', 'onehot': 0.5, 'log': False, 'scaler': 'robust', 'features': ('days_before_arrival', 'special_requests', 'nightly_rate_idr', 'booking_channel', 'doy_cos', 'total_nights', 'needs_parking', 'booking_index', 'weekend_nights', 'is_returning_guest', 'room_category', 'arrival_month'), 'fw': (('booking_index', 2.0), ('room_category', 0.25), ('total_nights', 0.5), ('weekend_nights', 0.5)), 'k': 13, 'p': 0.75, 'weight': 'inv', 'enn_k': 0, 'enn_mode': 'drop', 'threshold': 'oof'}
THRESHOLD = 0.46


,proportion
is_canceled,
0,0.6859
1,0.3141
